# 01 — Environment & Intelligence
## Agentic X | Studio 67

This notebook prepares the simulated event environment, validates the scenario dataset, defines agent and environment characteristics, and executes the Creative LLM Agent.

The system handles event crises through four collaborating agents: Creative, Finance, Operations, and Safety.


Cell 2 — Connect to GitHub

In [1]:

from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/ProjectsSpaceHub/AgenticX.git"
ROOT = Path("/content/AgenticX")

if not ROOT.exists():
    subprocess.run(
        ["git", "clone", REPO_URL, str(ROOT)],
        check=True
    )

os.chdir(ROOT)

print("Project directory:", Path.cwd())


Project directory: /content/AgenticX


## 2. Load and validate source data
Read all 150 incident cases and the candidate option records.

In [ ]:
cases, options = load_dataset()
print('Cases:',len(cases),'Options:',len(options))
print('Scenario distribution:',cases['scenario'].value_counts().to_dict())
assert len(cases)==150 and len(options)==450
assert options.groupby('case_id').size().eq(3).all()

## 3. Describe environment and classification
Discuss PEAS, event dynamics and why some inputs are incomplete. All four agents' PEAS should also be summarized in the final report.

In [ ]:
from pprint import pprint
pprint(ENVIRONMENT_DESCRIPTION)
PEAS_CREATIVE={'Performance':'Relevance and audience-quality score','Environment':'Studio 67 incident and candidate solutions','Actuators':'Propose / accept / refuse with justification','Sensors':'Incident text, option details, quality threshold'}
pprint(PEAS_CREATIVE)

## 4. Inspect a real scenario observation
Agent-facing option fields **exclude** evaluation-only `addresses_incident` and precomputed outcomes.

In [ ]:
case, candidates = get_case('SCR-001')
print(case['case_id'],case['scenario'],case['incident'])
print('Options:',[o['option_id'] for o in candidates])
assert all('addresses_incident' not in o for o in candidates)

## 5. Build real Creative LLM Agent
Requires `OPENAI_API_KEY` (optionally `OPENAI_BASE_URL`). This cell makes actual API requests; no fake fallback.

In [ ]:
creative = CreativeAgent()
creative_results=[creative.evaluate(case,option) for option in candidates]
for result in creative_results: print(result.model_dump())

## 6. Assertions and handoff to Notebook 02
Check the schema; score is descriptive, not a fabricated success metric.

In [ ]:
assert len(creative_results)==len(candidates)
assert {r.agent for r in creative_results}=={'Creative'}
assert {r.option_id for r in creative_results}=={o['option_id'] for o in candidates}
print('Creative evaluations produced by a live LLM.')